In [ ]:
import pandas as pd
import numpy as np
import os

DATA_DIR    = 'data'
ANNOT_DIR   = 'annotation'
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

BATCH_FILES = {
    'batch_01': (f'{ANNOT_DIR}/batch_01_A_filled.csv', f'{ANNOT_DIR}/batch_01_B_filled.csv', 'A', 'B'),
    'batch_02': (f'{ANNOT_DIR}/batch_02_B_filled.csv', f'{ANNOT_DIR}/batch_02_C_filled.csv', 'B', 'C'),
    'batch_03': (f'{ANNOT_DIR}/batch_03_C_filled.csv', f'{ANNOT_DIR}/batch_03_D_filled.csv', 'C', 'D'),
    'batch_04': (f'{ANNOT_DIR}/batch_04_D_filled.csv', f'{ANNOT_DIR}/batch_04_A_filled.csv', 'D', 'A'),
}

def cohen_kappa(a, b):
    n   = len(a)
    if n == 0: return None, None
    p_o = sum(x == y for x, y in zip(a, b)) / n
    p_a1, p_b1 = sum(a)/n, sum(b)/n
    p_e = p_a1*p_b1 + (1-p_a1)*(1-p_b1)
    if p_e == 1.0: return 1.0, p_o
    return round((p_o - p_e) / (1 - p_e), 4), round(p_o, 4)

def interpret(k):
    if k is None: return "N/A"
    if k < 0.20: return "Slight"
    if k < 0.40: return "Fair"
    if k < 0.60: return "Moderate"
    if k < 0.80: return "Substantial"
    return "Near-perfect"

kappas, all_disagreements = {}, []

for batch, (path_a, path_b, ann1, ann2) in BATCH_FILES.items():
    if not os.path.exists(path_a) or not os.path.exists(path_b):
        print(f"Skipping {batch} — files not found"); continue

    df_a = pd.read_csv(path_a)
    df_b = pd.read_csv(path_b)

    for df in [df_a, df_b]:
        df['score'] = pd.to_numeric(df['score'], errors='coerce')

    merged = pd.merge(
        df_a[['annotation_id','claim_id','framing','model_code','wrong_claim','response','score','confidence']],
        df_b[['annotation_id','score','confidence']],
        on='annotation_id', suffixes=(f'_{ann1}', f'_{ann2}')
    ).dropna(subset=[f'score_{ann1}', f'score_{ann2}'])

    sa = merged[f'score_{ann1}'].astype(int).tolist()
    sb = merged[f'score_{ann2}'].astype(int).tolist()

    k, p_o = cohen_kappa(sa, sb)
    pair    = f"{ann1}{ann2}"
    kappas[pair] = {'kappa': k, 'p_obs': p_o, 'n': len(merged)}
    print(f"{pair}: n={len(merged)} | kappa={k} ({interpret(k)}) | agreement={p_o:.1%}")

    disc = merged[merged[f'score_{ann1}'] != merged[f'score_{ann2}']].copy()
    disc['pair'] = pair
    disc['score_A'] = disc[f'score_{ann1}']
    disc['score_B'] = disc[f'score_{ann2}']
    all_disagreements.append(disc[['annotation_id','claim_id','framing','model_code',
                                    'wrong_claim','response','pair','score_A','score_B',
                                    f'confidence_{ann1}', f'confidence_{ann2}']])

print("\n=== SUMMARY ===")
vals = [v['kappa'] for v in kappas.values() if v['kappa'] is not None]
if vals:
    print(f"Mean kappa: {np.mean(vals):.4f} ({interpret(np.mean(vals))})")

if all_disagreements:
    disc_df = pd.concat(all_disagreements, ignore_index=True)
    print(f"\nTotal disagreements: {len(disc_df)}")
    print(f"\nBy framing:\n{disc_df['framing'].value_counts().to_string()}")
    print(f"\nBy model:\n{disc_df['model_code'].value_counts().to_string()}")

    adj = disc_df.copy()
    adj['adjudicator_score'] = ''
    adj['adjudicator_notes'] = ''
    adj.to_csv(f'{DATA_DIR}/adjudication_sheet.csv', index=False)
    print(f"\nAdjudication sheet → {DATA_DIR}/adjudication_sheet.csv")
